# House Stage 1: lagged Split Ticket WAR (feature ablation)

Does a race's nominees' past **Wins Above Replacement** (Split Ticket) improve Stage 1, on top of lean, the generic
ballot, incumbency and (in the ensemble) fundraising?

**Adopted 2026-11-03:** production House Stage 1 is now **85/15 + WAR** (`src/house/stage1.py`; the lag is
`modeling.csv`'s `war_f` tranche). This notebook keeps the earlier Model A, spelled out below as `MODEL_A_PRE_WAR`, as
its baseline, so the ablation stays reproducible.

**The predictor** (`src/congress/war.py`, data from `data-processing/split_ticket_war.py`):
- **WAR** is Split Ticket's residual of the congressional-minus-presidential margin against a per-cycle baseline that
  already controls for incumbency, lagged presidential partisanship and swing, spending and demographics. It belongs
  to the race (both candidates). `war_dem` is D positive ("D+3" = +3), centered by chamber-year.
- **Whose races:** every earlier House WAR race in which either current nominee was the D or R nominee, matched by
  FEC candidate ID (with re-registered IDs of one person linked). A past Republican over-performance counts against
  the Democrat now. A rematch counts once.
- **The lag:** one earlier year with WAR, that value; two or more, $0.75\,w_{recent} + 0.25\,w_{previous}$. Only
  years before the race, never its own.
- **Missing WAR is neutral**, as in the Senate lagged-performance experiment (`S1 + L (neutral)`): a race with no
  earlier WAR gets `war_lag = 0`, flagged `war_lag_available = False`, so **no training or scored row is dropped** and
  every model trains on exactly production's rows (asserted in section 2). The 0 means "no candidate signal", and
  WAR is centered, so 0 is the average candidate. No separate missing-WAR coefficient is estimated.

**Specifications** (one extra coefficient each, the WAR persistence, estimated by the same equal-year WLS):
- **Model A + WAR:** `partisan_lean + gen_margin_raw + 2.5·inc_signed + partisan_lean:inc_signed + war_lag`.
- **85/15 + WAR:** 0.85 × (Model A + WAR) + 0.15 × Model B, Model B unchanged.

Same training windows (2006 → the year before), same rows (no race dropped for a missing lag), same scored races,
same Stage 2 Monte Carlo. Nothing is tuned: the lag weights are fixed, and the ±10 cap in section 5 is a sensitivity
check only.

**What is not a clean out-of-sample test** (details in `data-processing/README.md`):
- One WAR vintage, refreshed May 2026. Split Ticket recomputed 2018–2022 in January 2025 with its 2024 method, so
  every past WAR used here was produced after 2024's results were known. Each cycle's baseline appears to be fit on
  that cycle's races alone (checked in section 1), so no later *results* should enter an earlier year's WAR, but the
  *method* was chosen with hindsight.
- No WAR existed in real time for most of these forecasts (2014–2020 WAR was published later). The 2014 rows are in
  the database file but on no page of Split Ticket's site.
- WAR starts in 2014, so the first lag is for 2016: the 2018 window learns the coefficient from 2016 alone.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr
from congress import war
from house import simulation, stage1

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
EVAL_YEARS = [2018, 2020, 2022, 2024]  # stage1.ROLLING_EVAL_YEARS
WAR_CAP = 10.0  # section 5 only: the lag clipped to +-10 points, a sensitivity check

with warnings.catch_warnings(action="ignore"):  # known optional-column gaps
    predictors = hr.load_predictors()  # modeling.csv, with war_lag (0 where missing) and war_lag_available
    targets = hr.load_targets()
war_races = war.load_war()
lags = war.chamber_lags("House")  # one row per district-year 2006-2026, recomputed from the WAR files
# modeling.csv's war_f tranche is exactly this lag, with the neutral 0
stored = predictors[[*hr.KEY, war.LAG, war.AVAILABLE]].merge(war.house_war_features(lags), on=hr.KEY,
                                                            suffixes=("", "_src"), validate="one_to_one")
assert len(stored) == len(predictors) and np.allclose(stored[war.LAG], stored[f"{war.LAG}_src"])
assert stored[war.AVAILABLE].equals(stored[f"{war.AVAILABLE}_src"])
predictors_war = predictors.merge(lags[[*hr.KEY, "war_from"]], on=hr.KEY, validate="one_to_one")

## 1. The WAR data

**Signs.** Known cases, D positive: Kaptur against Majewski (2022) and Peltola against Palin (2022) were famous
Democratic over-performances; Bacon (NE-2) and Fitzpatrick (PA-1) are Republicans who run ahead of their districts.

In [2]:
examples = [(2022, "OH-09"), (2022, "AK-AL"), (2024, "ME-02"), (2022, "NE-02"), (2024, "PA-01"), (2022, "NY-17")]
house_war = war_races[war_races["chamber"] == "House"]
house_war.set_index(["year", "geography"]).loc[examples, ["war_democrat", "war_republican", "war_label", "war_dem"]]

war_democrat     war_republican war_label  war_dem
year geography                                                            
2022 OH-09              Marcy Kaptur        JR Majewski    D+15.6     15.6
     AK-AL              Mary Peltola        Sarah Palin    D+22.1     22.1
2024 ME-02              Jared Golden   Austin Theriault     D+5.9      5.9
2022 NE-02               Tony Vargas          Don Bacon       R+3     -3.0
2024 PA-01              Ashley Ehasz  Brian Fitzpatrick     R+8.8     -8.8
2022 NY-17      Sean Patrick Maloney        Mike Lawler     R+1.7     -1.7

**Is WAR already net of incumbency and money?** If each cycle's baseline is a regression on that cycle's races with
incumbency among its controls, WAR should average 0 within each year and be uncorrelated with signed incumbency. The
table regresses WAR on our `inc_signed`, `fundraising_advantage` and `partisan_lean`, year by year.

In [3]:
checks = house_war.merge(predictors[[*hr.KEY, "inc_signed", "fundraising_advantage", "partisan_lean"]],
                         on=hr.KEY, how="left", validate="one_to_one")
rows = {}
for year, g in checks.groupby("year"):
    fit = smf.ols("war_dem ~ inc_signed + fundraising_advantage + partisan_lean", data=g).fit()
    rows[year] = {"races": len(g), "mean": g["war_dem"].mean(), "sd": g["war_dem"].std(),
                  **{f"coef {k}": fit.params[k] for k in ["inc_signed", "fundraising_advantage", "partisan_lean"]},
                  "se inc_signed": fit.bse["inc_signed"]}
pd.DataFrame(rows).T.round(2)

,races,mean,sd,coef inc_signed,coef fundraising_advantage,coef partisan_lean,se inc_signed
2014,357.0,-0.11,8.58,-0.17,1.90,-0.24,0.84
2016,369.0,0.02,7.34,-0.02,1.72,-0.23,0.71
2018,394.0,0.23,5.45,1.35,0.82,-0.12,0.48
2020,408.0,-0.15,4.66,0.86,0.89,-0.11,0.36
2022,400.0,0.15,4.43,-0.00,0.75,-0.07,0.42
2024,398.0,0.20,3.88,-0.53,0.85,-0.08,0.35


- **Mean 0 each year:** consistent with a per-cycle fit, so a year's WAR shouldn't contain later results.
- **Incumbency:** the coefficient is near 0 in 2014, 2016, 2022 and 2024, and about +1 in 2018 and 2020. Most of
  incumbency is already out of WAR, so the lag doesn't double-count Model A's fixed +2.5.
- **Money:** WAR still rises with our `fundraising_advantage` (about 0.8–1.9 per log unit). Split Ticket's spending
  control is coarse (a ratio only when one side passes \$1.5M), so part of a candidate's WAR is fundraising, which
  Model B also uses. Section 4 checks whether the gain survives the 85/15 blend.

**Coverage of the lag.** Every district-year, by what the lag is built from. Scored races only further down.

In [4]:
flags = lags.assign(flag=lags["war_flag"].str.replace(r" \(.*\)", "", regex=True))
display(pd.crosstab(flags["year"], flags["flag"]).loc[2014:])
display(Markdown("**Whose earlier races** (dem: the Democrat's; gop: the Republican's; both: a rematch or each in "
                 "their own race)"))
display(pd.crosstab(flags["year"], flags["war_from"].fillna("none")).loc[2016:])
display(Markdown("**Party switchers** (the lag flips sign)"))
lags.loc[lags["war_party_switch"], [*hr.KEY, "dem_name", "gop_name", war.LAG, "war_flag"]].round(2)

flag,neutral: no earlier WAR race of either nominee,one year,two years
year,,,
2014,435,0,0
2016,121,314,0
2018,92,116,227
2020,51,124,260
2022,69,65,301
2024,41,93,301
2026,77,73,285


**Whose earlier races** (dem: the Democrat's; gop: the Republican's; both: a rematch or each in their own race)

war_from,both,dem,gop,none
year,,,,
2016,59,103,152,121
2018,36,130,177,92
2020,65,186,133,51
2022,66,133,167,69
2024,82,148,164,41
2026,54,151,153,77


**Party switchers** (the lag flips sign)

,year,state,district,dem_name,gop_name,war_lag,war_flag
3298,2020,NJ,2,AMY KENNEDY,JEFFERSON VAN DREW,-0.07,one year (2018)
3734,2022,NJ,2,TIM ALEXANDER,JEFFERSON VAN DREW,0.40,"two years (2020, 2018)"
4169,2024,NJ,2,JOSEPH F. SALERNO,JEFFERSON VAN DREW,-3.72,"two years (2022, 2020)"
4604,2026,NJ,2,Zack Mullock,Jeff Van Drew,-2.63,"two years (2024, 2022)"


In [5]:
display(Markdown("**The largest lags** (|lag| > 15), 2016-2026"))
big = lags[lags[war.LAG].abs() > 15].sort_values(war.LAG)
big[[*hr.KEY, "dem_name", "gop_name", war.LAG, "war_flag", "war_from"]].round(2)

**The largest lags** (|lag| > 15), 2016-2026

,year,state,district,dem_name,gop_name,war_lag,war_flag,war_from
2495,2016,OR,2,"JAMES \""JIM\"" CRARY",GREG WALDEN,-26.69,one year (2014),gop
2700,2018,FL,4,"GEORGE \""GES\"" SELMONT",JOHN H RUTHERFORD,-23.92,one year (2016),gop
2447,2016,NY,2,DUWAYNE GREGORY,PETER T KING,-23.39,one year (2014),gop
2469,2016,NY,24,COLLEEN DEACON,JOHN KATKO,-22.49,one year (2014),gop
2904,2018,NY,24,DANA BALTER,JOHN M KATKO,-22.14,"two years (2016, 2014)",gop
3339,2020,NY,24,DANA BALTER,JOHN M KATKO,-19.86,"two years (2018, 2016)",both
2761,2018,IL,16,SARA DADY,ADAM KINZINGER,-19.29,one year (2014),gop
2326,2016,IL,16,NaN,ADAM KINZINGER,-19.29,one year (2014),gop
2502,2016,PA,4,JOSHUA BURKHOLDER,SCOTT PERRY,-18.99,one year (2014),gop
2234,2016,CA,39,BRETT MURDOCK,EDWARD R ROYCE,-18.59,one year (2014),gop


## 2. Fits

The earlier Model A and 85/15 ensemble (`MODEL_A_PRE_WAR`: production until 2026-11-03), and the same two with
`war_lag` in Model A, which is today's production (`stage1.HOUSE_MODEL_A`, `stage1.HOUSE_ENSEMBLE_AB_85_15`).

In [6]:
MODEL_A_PRE_WAR = stage1.Component(  # production Model A until 2026-11-03
    "A_polling",
    f"{stage1.TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed",
    fixed={"inc_signed": 2.5},
)
assert stage1.MODEL_A.formula == f"{MODEL_A_PRE_WAR.formula} + war_lag"  # production = the baseline + war_lag
CONFIGS = {
    "A": stage1.Stage1Config("house_model_a_pre_war", ((MODEL_A_PRE_WAR, 1.0),)),
    "A + WAR": stage1.HOUSE_MODEL_A,
    "85/15": stage1.Stage1Config("house_ensemble_ab_85_15_pre_war", ((MODEL_A_PRE_WAR, 0.85), (stage1.MODEL_B, 0.15))),
    "85/15 + WAR": stage1.HOUSE_ENSEMBLE_AB_85_15,
}
with warnings.catch_warnings(action="ignore"):
    rolling = {name: stage1.rolling_predictions(config, EVAL_YEARS, predictors=predictors_war, targets=targets)
               for name, config in CONFIGS.items()}


def train_keys(result, component):
    return result.components[component].predictions.query("role == 'train'")[hr.KEY].reset_index(drop=True)


# Neutral missing WAR: every WAR model trains on exactly production's rows, and predicts the same races
for base, with_war in [("A", "A + WAR"), ("85/15", "85/15 + WAR")]:
    for year in EVAL_YEARS:
        assert train_keys(rolling[base][year], "A_polling").equals(train_keys(rolling[with_war][year], "A_polling"))
        assert rolling[base][year].predictions[hr.KEY].equals(rolling[with_war][year].predictions[hr.KEY])
print("No training or predicted row dropped for a missing lag.")

No training or predicted row dropped for a missing lag.


## 3. The WAR persistence coefficient

By training window (the eval year's). `war races` is the training races with a real lag; the rest carry 0.

In [7]:
coef_rows = {}
for year in EVAL_YEARS:
    run = rolling["A + WAR"][year].components["A_polling"]
    train = run.predictions.query("role == 'train'").merge(predictors_war[[*hr.KEY, war.AVAILABLE]], on=hr.KEY)
    coef_rows[year] = {**run.coefficients["coef"].to_dict(), "se war_lag": run.coefficients.loc["war_lag", "std_err"],
                       "train races": len(train), "war races": int(train[war.AVAILABLE].sum())}
pd.DataFrame(coef_rows).T.rename_axis("eval year").round(3)

,Intercept,partisan_lean,gen_margin_raw,partisan_lean:inc_signed,war_lag,inc_signed,se war_lag,train races,war races
eval year,,,,,,,,,
2018,1.944,1.099,0.793,-0.124,0.667,2.5,0.093,2282.0,280.0
2020,1.844,1.072,0.825,-0.107,0.617,2.5,0.065,2675.0,593.0
2022,1.565,1.054,0.791,-0.106,0.611,2.5,0.055,3083.0,958.0
2024,1.453,1.043,0.776,-0.093,0.596,2.5,0.049,3482.0,1292.0


**Leave one training year out**, on the 2026 window (2006–2024): the coefficient with each year dropped in turn.
Only 2016–2024 carry lags, so those are the years that can move it.

In [8]:
WINDOW_2026 = stage1.expanding_window(CONFIGS["A + WAR"], 2026)
loyo = {}
with warnings.catch_warnings(action="ignore"):
    for dropped in [None, *range(2016, 2026, 2)]:
        years = [y for y in WINDOW_2026 if y != dropped]
        fit = stage1.fit_predict(CONFIGS["A + WAR"], years, 2026, predictors=predictors_war, targets=targets)
        loyo["none" if dropped is None else dropped] = fit.components["A_polling"].coefficients["coef"]
pd.DataFrame(loyo).T.rename_axis("dropped").round(3)

term,Intercept,partisan_lean,gen_margin_raw,partisan_lean:inc_signed,war_lag,inc_signed
dropped,,,,,,
none,1.226,1.032,0.783,-0.088,0.590,2.5
2016,1.644,1.022,0.775,-0.098,0.562,2.5
2018,1.192,1.042,0.743,-0.094,0.615,2.5
2020,1.380,1.041,0.821,-0.086,0.591,2.5
2022,1.291,1.040,0.798,-0.099,0.604,2.5
2024,1.453,1.043,0.776,-0.093,0.596,2.5


## 4. Out-of-sample comparison, 2018–2024

The same scored races for every model: the ensemble's valid targets (Model A alone scores a few more districts,
which lack `fundraising_advantage`; they are left out here so every row is predicted by all four). Subsets:
- **ex-ante competitive:** |lean + eve generic ballot| ≤ 10 (Stage 1's `competitive_ex_ante`, known before the
  election).
- **incumbent / open:** `inc_signed` ≠ 0 / = 0.
- **with / without a lag:** `war_lag_available`.

`bias` = mean(actual − prediction), so negative is too Democratic. `winners_called_pct` counts races with a nonzero
prediction and result.

In [9]:
# the ensemble's scored races: Model A alone also scores a few districts the ensemble can't (no fundraising_advantage)
scored = pd.concat([r.scored for r in rolling["85/15"].values()], ignore_index=True)
oos = scored[[*hr.KEY, "actual_margin", "competitive_ex_ante", "competitive_ex_post"]].rename(
    columns={"actual_margin": "actual"})
for name, run in rolling.items():
    predicted = pd.concat([r.predictions for r in run.values()], ignore_index=True)
    oos = oos.merge(predicted[[*hr.KEY, "prediction"]].rename(columns={"prediction": name}), on=hr.KEY, how="left",
                    validate="one_to_one")
oos = oos.merge(predictors_war[[*hr.KEY, "inc_signed", war.LAG, war.AVAILABLE, "war_from"]], on=hr.KEY,
                validate="one_to_one")
assert oos[list(CONFIGS)].notna().all().all()  # identical scored races for every model

incumbent, ex_ante, has_lag = oos["inc_signed"] != 0, oos["competitive_ex_ante"], oos[war.AVAILABLE]
SUBSETS = {
    "all": pd.Series(True, index=oos.index),
    "ex-ante competitive": ex_ante,
    "incumbent": incumbent,
    "open": ~incumbent,
    "ex-ante competitive incumbent": ex_ante & incumbent,
    "ex-ante competitive open": ex_ante & ~incumbent,
    "with a lag": has_lag,
    "without a lag": ~has_lag,
    "ex-ante competitive with a lag": ex_ante & has_lag,
}
MODELS = {name: name for name in CONFIGS}
pooled = war.score(oos, MODELS, SUBSETS)
pooled.unstack("model").reindex(SUBSETS).round(2)

n                             mae                            bias                           winners_called_pct                           
model                          85/15 85/15 + WAR     A A + WAR 85/15 85/15 + WAR     A A + WAR 85/15 85/15 + WAR     A A + WAR              85/15 85/15 + WAR      A A + WAR
subset                                                                                                                                                                      
all                             1597        1597  1597    1597  5.20        4.80  5.48    5.07 -0.44       -0.44 -0.28   -0.27              96.12       96.62  96.12   96.31
ex-ante competitive              353         353   353     353  4.70        4.28  4.99    4.60 -0.50       -0.44 -0.30   -0.23              86.12       88.10  86.12   86.97
incumbent                       1371        1371  1371    1371  5.31        4.85  5.60    5.12 -0.42       -0.37 -0.21   -0.15              96.57       97.08  96.57   96.72
open                             226         226   226     226  4.53        4.48  4.74    4.73 -0.59       -0.84 -0.75   -1.04              93.36       93.81  93.36   93.81
ex-ante competitive incumbent    288         288   288     288  4.71        4.24  5.03    4.60 -0.65       -0.49 -0.44   -0.25              87.85       89.93  87.85   88.54
ex-ante competitive open          65          65    65      65  4.66        4.47  4.79    4.59  0.14       -0.23  0.28   -0.16              78.46       80.00  78.46   80.00
with a lag                      1374        1374  1374    1374  5.28        4.81  5.54    5.04 -0.47       -0.46 -0.29   -0.28              96.29       96.87  96.29   96.51
without a lag                    223         223   223     223  4.69        4.76  5.13    5.22 -0.26       -0.28 -0.24   -0.27              95.07       95.07  95.07   95.07
ex-ante competitive with a lag   305         305   305     305  4.76        4.27  5.07    4.62 -0.67       -0.59 -0.49   -0.39              86.89       89.18  86.89   87.87

### By election year

In [10]:
by_year = pd.concat({year: war.score(oos, MODELS, {"all": oos["year"] == year,
                                                   "ex-ante competitive": (oos["year"] == year) & ex_ante})
                     for year in EVAL_YEARS}, names=["year"])
by_year[["n", "mae", "bias", "winners_called_pct"]].unstack("model").round(2)

n                            mae                            bias                           winners_called_pct                           
model                    85/15 85/15 + WAR    A A + WAR 85/15 85/15 + WAR     A A + WAR 85/15 85/15 + WAR     A A + WAR              85/15 85/15 + WAR      A A + WAR
year subset                                                                                                                                                          
2018 all                   393         393  393     393  5.62        5.02  6.28    5.71  1.24        1.33  2.37    2.47              95.17       95.42  94.91   95.17
     ex-ante competitive   100         100  100     100  4.56        4.37  5.12    5.12  1.92        2.38  2.96    3.50              84.00       84.00  83.00   83.00
2020 all                   408         408  408     408  5.34        4.78  5.74    5.10 -3.05       -3.11 -3.15   -3.22              96.57       96.81  96.57   96.32
     ex-ante competitive    92          92   92      92  4.67        4.07  4.98    4.33 -2.30       -2.45 -2.36   -2.54              91.30       91.30  91.30   90.22
2022 all                   398         398  398     398  5.22        5.11  5.21    5.07  1.57        1.58  0.83    0.84              95.23       95.98  95.23   95.48
     ex-ante competitive    81          81   81      81  5.13        4.80  5.25    5.01 -0.06       -0.09 -0.88   -0.90              79.01       82.72  79.01   80.25
2024 all                   398         398  398     398  4.61        4.30  4.70    4.38 -1.46       -1.46 -1.08   -1.08              97.49       98.24  97.74   98.24
     ex-ante competitive    80          80   80      80  4.47        3.89  4.58    3.85 -1.90       -2.02 -1.44   -1.58              90.00       95.00  91.25   95.00

### By whose earlier races the lag comes from

Races with a lag only. `dem` / `gop`: only that nominee has earlier WAR races (usually the incumbent); `both`: a
rematch, or both nominees ran before.

In [11]:
by_source = war.score(oos, MODELS, {src: oos["war_from"] == src for src in ["dem", "gop", "both"]})
by_source[["n", "mae", "bias"]].unstack("model").round(2)

n                            mae                            bias                          
model  85/15 85/15 + WAR    A A + WAR 85/15 85/15 + WAR     A A + WAR 85/15 85/15 + WAR     A A + WAR
subset                                                                                               
both     249         249  249     249  5.95        5.20  6.09    5.20 -1.93       -1.68 -1.83   -1.54
dem      536         536  536     536  5.07        4.71  5.18    4.80 -2.45       -2.84 -2.28   -2.74
gop      589         589  589     589  5.20        4.73  5.63    5.19  1.94        2.22  2.17    2.50

## 5. Sensitivity to large WAR values

Lags beyond ±10 points (Peltola, Kaptur 2024, ...) carry the most leverage. The same two WAR specifications with the
lag clipped to ±`WAR_CAP` before fitting and predicting. A check only, not a candidate specification.

In [12]:
predictors_capped = war.with_war_lag(predictors.drop(columns=[war.LAG, war.AVAILABLE]), lags, hr.KEY, cap=WAR_CAP)
with warnings.catch_warnings(action="ignore"):
    capped = {f"{name} (cap {WAR_CAP:g})": stage1.rolling_predictions(CONFIGS[name], EVAL_YEARS,
                                                                       predictors=predictors_capped, targets=targets)
              for name in ["A + WAR", "85/15 + WAR"]}
for name, run in capped.items():
    predicted = pd.concat([r.predictions for r in run.values()], ignore_index=True)
    oos = oos.merge(predicted[[*hr.KEY, "prediction"]].rename(columns={"prediction": name}), on=hr.KEY,
                    validate="one_to_one")
large = oos[war.LAG].abs() > WAR_CAP
display(Markdown(f"Scored races with |lag| > {WAR_CAP:g}: **{int(large.sum())}** of {len(oos)}. "
                 f"Coefficient with the cap, by window: "
                 + ", ".join(f"{y}: {r.components['A_polling'].coefficients.loc['war_lag', 'coef']:.3f}"
                             for y, r in capped["A + WAR (cap 10)"].items())))
war.score(oos, {**MODELS, **{n: n for n in capped}},
          {"all": SUBSETS["all"], "ex-ante competitive": ex_ante, f"|lag| > {WAR_CAP:g}": large}).unstack(
    "model")[["mae", "bias"]].round(2)

Scored races with |lag| > 10: **95** of 1597. Coefficient with the cap, by window: 2018: 0.891, 2020: 0.793, 2022: 0.755, 2024: 0.721

mae                                                                  bias                                                                
model               85/15 85/15 + WAR 85/15 + WAR (cap 10)     A A + WAR A + WAR (cap 10) 85/15 85/15 + WAR 85/15 + WAR (cap 10)     A A + WAR A + WAR (cap 10)
subset                                                                                                                                                         
all                  5.20        4.80                 4.79  5.48    5.07             5.07 -0.44       -0.44                -0.43 -0.28   -0.27            -0.26
ex-ante competitive  4.70        4.28                 4.29  4.99    4.60             4.64 -0.50       -0.44                -0.49 -0.30   -0.23            -0.29
|lag| > 10           8.45        6.04                 6.03  9.21    6.57             6.54 -0.09       -0.28                 0.05  0.43    0.20             0.59

## 6. Stage 2: the Monte Carlo backtest

The production Monte Carlo (`house.simulation`, H0), exactly as in `02_monte_carlo.ipynb`: for each test year the
error SDs are refit on the earlier years' out-of-sample residuals **of the configuration being tested**, every race is
simulated under the race rules, and the draws are scored. Each WAR ensemble falls back to Model A + WAR. Same seed and
draws for every configuration.

In [13]:
SEED = 42
N_SIMS = 40_000
NU = 5
NATIONAL_PRIOR_SD = 3.0
NATIONAL_PRIOR_ELECTIONS = 2
CALIBRATION_YEARS = [2020, 2022, 2024]
FALLBACK = {"A": "A", "A + WAR": "A + WAR", "85/15": "A", "85/15 + WAR": "A + WAR"}

targets_raw = simulation.load_targets_raw()
races = simulation.historical_races(targets, targets_raw)
stage1_runs = {name: simulation.stage1_inputs(config, predictors=predictors_war, targets=targets)
               for name, config in CONFIGS.items()}
calibration = {}
for name, run in stage1_runs.items():
    calibration[name] = simulation.backtest(
        run.oos,
        test_years=CALIBRATION_YEARS,
        races=races,
        predictions=run.predictions,
        fallback_predictions=stage1_runs[FALLBACK[name]].predictions,
        nu=NU,
        n_sims=N_SIMS,
        seed=SEED,
        house_size=435,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
    )
for name, cal in calibration.items():
    print(simulation.check_backtest(cal, stage1_runs[name], races, house_size=435, n_sims=N_SIMS,
                                    national_prior_sd=NATIONAL_PRIOR_SD,
                                    national_prior_elections=NATIONAL_PRIOR_ELECTIONS))

backtest checks passed: house_model_a_pre_war (national, state, district), 2020, 2022, 2024


backtest checks passed: house_model_a (national, state, district), 2020, 2022, 2024


backtest checks passed: house_ensemble_ab_85_15_pre_war (national, state, district), 2020, 2022, 2024


backtest checks passed: house_ensemble_ab_85_15 (national, state, district), 2020, 2022, 2024


In [14]:
display(Markdown("**Pooled over 2020-2024** (race-level; seat counts are not pooled)"))
display(pd.DataFrame({name: cal.pooled() for name, cal in calibration.items()}).T.round(4))
table = simulation.stack(calibration, "table")
columns = [c for c in table.columns if any(k in c for k in ["Brier", "coverage", "seat", "actual"])]
table[columns]

**Pooled over 2020-2024** (race-level; seat counts are not pooled)

,races scored,Brier (all races),contested Brier,competitive Brier,observed margins,80% coverage,80% mean width,95% coverage,95% mean width
A,1305.0,0.0271,0.0293,0.1020,1205.0,0.8515,18.9528,0.9701,31.3030
A + WAR,1305.0,0.0237,0.0256,0.0867,1205.0,0.8382,17.1747,0.9685,28.3606
85/15,1305.0,0.0266,0.0288,0.0997,1205.0,0.8465,17.5986,0.9651,29.0446
85/15 + WAR,1305.0,0.0235,0.0255,0.0859,1205.0,0.8290,15.9613,0.9618,26.3232


Brier (all races)  contested Brier  competitive Brier  80% coverage  95% coverage  median panel_dem_seats 80% panel seat interval 95% panel seat interval  panel_actual_dem_seats  \
config      test year                                                                                                                                                                                      
A           2020                0.030826         0.032865           0.085951      0.833333      0.960784                     231                 219–243                 211–252                     222   
            2022                0.033888         0.036853           0.151297      0.849624      0.982456                     212                 199–224                 190–231                     213   
            2024                0.016522         0.018058           0.070429      0.871859      0.967337                     215                 204–225                 196–231                     215   
A + WAR     2020                0.028907         0.030819           0.079244      0.840686      0.963235                     233                 222–244                 214–252                     222   
            2022                0.028857         0.031382           0.127419      0.814536      0.974937                     213                 200–224                 191–231                     213   
            2024                0.013334         0.014573           0.053950      0.859296      0.967337                     216                 206–226                 198–232                     215   
85/15       2020                0.030096         0.032088           0.083451      0.848039      0.963235                     231                 220–242                 213–250                     222   
            2022                0.033908         0.036875           0.150430      0.832080      0.979950                     210                 197–221                 189–228                     213   
            2024                0.015902         0.017380           0.067106      0.859296      0.952261                     216                 206–225                 199–231                     215   
85/15 + WAR 2020                0.028017         0.029871           0.076866      0.840686      0.960784                     233                 222–243                 215–250                     222   
            2022                0.029421         0.031995           0.129065      0.789474      0.972431                     210                 199–221                 190–228                     213   
            2024                0.013163         0.014387           0.052643      0.856784      0.952261                     217                 208–226                 200–232                     215   

                       actual-seat percentile  
config      test year                          
A           2020                     15.22375  
            2022                     54.45000  
            2024                     52.25500  
A + WAR     2020                      9.71875  
            2022                     51.86125  
            2024                     43.49000  
85/15       2020                     14.02375  
            2022                     65.30250  
            2024                     45.55375  
85/15 + WAR 2020                      9.11750  
            2022                     62.74750  
            2024                     36.38500

**Competitive seats.** Calls, Brier and interval coverage for the ex-ante competitive races, by test year.

In [15]:
stats = simulation.stack(calibration, "competitive_stats")
stats.xs("competitive_ex_ante", level="subset").round(3)

n  called D, won D  called D, won R  called R, won D  called R, won R  any other  wrong calls  accuracy  Brier  80% coverage  95% coverage
config      test year                                                                                                                                             
A           2020       92               35                7                1               49          0            8     0.913  0.086         0.880         0.989
            2022       81               43               10                7               21          0           17     0.790  0.151         0.827         0.975
            2024       80               44                3                4               29          0            7     0.912  0.070         0.888         0.975
A + WAR     2020       92               35                8                1               48          0            9     0.902  0.079         0.913         1.000
            2022       81               44               10                6               21          0           16     0.802  0.127         0.852         0.963
            2024       80               47                3                1               29          0            4     0.950  0.054         0.875         0.975
85/15       2020       92               35                7                1               49          0            8     0.913  0.083         0.902         0.989
            2022       81               42                9                8               22          0           17     0.790  0.150         0.827         0.975
            2024       80               44                4                4               28          0            8     0.900  0.067         0.875         0.975
85/15 + WAR 2020       92               35                7                1               49          0            8     0.913  0.077         0.902         1.000
            2022       81               44                8                6               23          0           14     0.827  0.129         0.877         0.963
            2024       80               47                3                1               29          0            4     0.950  0.053         0.875         0.975

## 7. What it does to 2026 (Stage 1 only)

The 85/15 ensemble trained on 2006–2024, with (production since 2026-11-03) and without the lag. 2026 nominees' lags come from their own earlier
House races (2024 and 2022 mostly). Only D-vs-R races are listed below; same-party and uncontested seats are decided
by rule in Stage 2.

In [16]:
with warnings.catch_warnings(action="ignore"):
    f_base = stage1.forecast(CONFIGS["85/15"], predictors=predictors_war, targets=targets).predictions
    f_war = stage1.forecast(CONFIGS["85/15 + WAR"], predictors=predictors_war, targets=targets).predictions
shift = f_base[[*hr.KEY, "prediction"]].merge(f_war[[*hr.KEY, "prediction"]], on=hr.KEY, suffixes=("_base", "_war"),
                                              validate="one_to_one")
shift = shift.merge(lags[[*hr.KEY, "dem_name", "gop_name", war.LAG, "war_flag"]], on=hr.KEY, validate="one_to_one")
shift["change"] = shift["prediction_war"] - shift["prediction_base"]
d_vs_r = shift["dem_name"].notna() & shift["gop_name"].notna()
shift = shift[d_vs_r]
flips = np.sign(shift["prediction_base"]) != np.sign(shift["prediction_war"])
display(Markdown(f"D-vs-R races: {len(shift)}. Mean change {shift['change'].mean():+.2f} points, mean |change| "
                 f"{shift['change'].abs().mean():.2f}. Stage 1 calls that flip: **{int(flips.sum())}** "
                 f"({int(((shift['prediction_base'] < 0) & flips).sum())} to D, "
                 f"{int(((shift['prediction_base'] > 0) & flips).sum())} to R). "
                 f"Predicted D seats among them: {int((shift['prediction_base'] > 0).sum())} -> "
                 f"{int((shift['prediction_war'] > 0).sum())}."))
display(Markdown("**Calls that flip**"))
display(shift[flips].sort_values("change")[[*hr.KEY, "dem_name", "gop_name", "prediction_base", "prediction_war",
                                             war.LAG, "war_flag"]].round(2))
display(Markdown("**Races within 5 points (either model), largest changes**"))
close = shift[(shift["prediction_base"].abs() <= 5) | (shift["prediction_war"].abs() <= 5)]
close.reindex(close["change"].abs().sort_values(ascending=False).index).head(15)[
    [*hr.KEY, "dem_name", "gop_name", "prediction_base", "prediction_war", war.LAG, "war_flag"]].round(2)

D-vs-R races: 417. Mean change -0.02 points, mean |change| 1.19. Stage 1 calls that flip: **4** (2 to D, 2 to R). Predicted D seats among them: 210 -> 210.

**Calls that flip**

,year,state,district,dem_name,gop_name,prediction_base,prediction_war,war_lag,war_flag
421,2026,WI,1,Mitchell Berman,Bryan Steil,0.72,-0.62,-2.66,"two years (2024, 2022)"
399,2026,VA,1,Shannon Taylor,Rob Wittman,0.05,-0.06,-0.18,"two years (2024, 2022)"
423,2026,WI,3,Rebecca Cooke,Derrick Van Orden,-1.39,1.30,5.42,"two years (2024, 2022)"
128,2026,IA,1,Christina Bohannan,Mariannette Miller-Meeks,-1.90,2.43,8.69,"two years (2024, 2022)"


**Races within 5 points (either model), largest changes**

,year,state,district,dem_name,gop_name,prediction_base,prediction_war,war_lag,war_flag
304,2026,OH,10,Kristina Knickerbocker,Mike Turner,-2.95,-8.18,-10.33,"two years (2024, 2022)"
321,2026,PA,1,Bob Harvie,Brian Fitzpatrick,6.46,1.61,-9.76,"two years (2024, 2022)"
384,2026,TX,28,Henry Cuellar,Tano Tijerina,4.76,9.27,8.99,"two years (2024, 2022)"
128,2026,IA,1,Christina Bohannan,Mariannette Miller-Meeks,-1.90,2.43,8.69,"two years (2024, 2022)"
303,2026,OH,9,Marcy Kaptur,Derek Merrin,1.53,5.18,7.32,"two years (2024, 2022)"
330,2026,PA,10,Janelle Stelson,Scott Perry,0.48,3.82,6.69,"two years (2024, 2022)"
96,2026,FL,13,Leela Gray,Anna Paulina Luna,-7.28,-4.10,6.49,"two years (2024, 2022)"
72,2026,CO,3,Dwayne Romero,Jeff Hurd,-4.88,-1.87,6.10,one year (2024)
423,2026,WI,3,Rebecca Cooke,Derrick Van Orden,-1.39,1.30,5.42,"two years (2024, 2022)"
130,2026,IA,3,Sarah Trone Garriott,Zach Nunn,2.20,4.08,3.74,"two years (2024, 2022)"


## Conclusion

**Lagged WAR adds real predictive value to House Stage 1, and the gain survives the 85/15 ensemble (with Model B's
fundraising) and the Monte Carlo.** It is the largest single-term gain these backtests have shown. **Adopted on
2026-11-03:** production House Stage 1 is 85/15 + WAR (the Senate keeps S1). The vintage caveat below is the main
reason for caution.

**1. A stable coefficient.** 0.67, 0.62, 0.61, 0.60 across the four windows (SE 0.05–0.09). Dropping one training
year from the 2026 window moves it within 0.56–0.62. An incumbent whose last two races beat the baseline by 10 points
is predicted about 6 points better. Split Ticket's own figure is "roughly half".

**2. Every year improves, competitive races included.** Pooled 2018–2024, the same 1,597 scored races:

| | 85/15 | 85/15 + WAR | A | A + WAR |
|---|---|---|---|---|
| MAE / bias | 5.20 / −0.44 | **4.80** / −0.44 | 5.48 / −0.28 | 5.07 / −0.27 |
| Ex-ante competitive MAE (353) | 4.70 | **4.28** | 4.99 | 4.60 |
| Ex-ante competitive winners called | 86.1% | **88.1%** | 86.1% | 87.0% |
| Ex-ante competitive incumbents MAE (288) | 4.71 | **4.24** | 5.03 | 4.60 |
| Open seats MAE (226) | 4.53 | 4.48 | 4.74 | 4.73 |
| Races without a lag (223) | 4.69 | 4.76 | 5.13 | 5.22 |

- **By year:**
  - Ensemble MAE: 5.62 → 5.02 (2018), 5.34 → 4.78 (2020), 5.22 → 5.11 (2022), 4.61 → 4.30 (2024).
  - Ex-ante competitive: 4.56 → 4.37, 4.67 → 4.07, 5.13 → 4.80, 4.47 → 3.89.
  - The weakest window is 2018, which learned the coefficient from 2016's lags alone: Model A's competitive MAE is
    flat there (5.12).
- **Where the gain is:** races with an earlier WAR race, mostly incumbents. Open seats barely move, and their bias
  goes 0.25 points more Democratic. Races without a lag lose 0.07, because the refit moves the other coefficients
  slightly.
- **Not driven by outliers:** clipping the lag at ±10 gives the same results (ensemble 4.79). On the 95 races with
  |lag| > 10, the MAE falls from 8.45 to 6.04, so the large WAR values carry real signal.

**3. It survives Stage 2.** The Monte Carlo backtest 2020–2024, with each configuration's error SDs refit on its own
residuals:
- **Brier:** all races 0.0266 → 0.0235; contested 0.0288 → 0.0255; ex-ante competitive 0.0997 → **0.0859**.
- **Competitive calls:** wrong calls fall from 33 to 26 (8/17/8 → 8/14/4), and Brier improves in every year.
- **Intervals narrow:** the 80% width goes from 17.6 to 16.0 points. 80% coverage is 0.829 (from 0.847) and still
  above nominal pooled, but 2022 falls to 0.789. 95% coverage is 0.962.
- **Seats:** 2020's actual count sits lower in the simulated distribution (percentile 14 → 9), so the national error
  stays the dominant seat risk, as before.

**4. What it does to 2026's Stage 1** (before margin adjustments).
- **Aggregate:** across the 417 D-vs-R races the mean change is −0.02 points and the mean absolute change 1.19, so
  the national picture is unchanged (210 predicted D wins either way).
- **Calls that flip:**
  - To D: IA-1 (Miller-Meeks, −1.9 → +2.4) and WI-3 (Van Orden, −1.4 → +1.3).
  - To R: WI-1 (Steil, +0.7 → −0.6) and VA-1 (Wittman, a coin flip either way).
- **Largest moves:** OH-10 (Turner) −5.2, PA-1 (Fitzpatrick) −4.9, TX-28 (Cuellar) +4.5, OH-9 (Kaptur) +3.6.
- These are the candidate histories (Van Orden's 2024, Miller-Meeks' repeated underperformance) the model was
  missing, now coming from data rather than overrides.

**Why it works:**
- Stage 1 has no candidate information beyond incumbency.
- WAR is a cleaner past residual than our own: it is net of incumbency, demographics and money, scored against
  same-year presidential results.
- It persists for the same person.

The House has about 400 races a year and no race-level polling, so nothing else in the pipeline carries this signal.

**Limitations:**
- **Not a clean out-of-sample test.**
  - Split Ticket recomputed 2018–2022 in 2025 with a method chosen after 2024, and 2014–2020 WAR did not exist when
    those elections were forecast.
  - The per-cycle fit (mean 0 each year) means no later *results* should enter an earlier year's WAR. But a method
    picked with hindsight can still flatter the backtest.
  - At best the latest forecasts could have used WAR in real time, and then only in its original vintage (the
    2022 House WAR was published in December 2022, before its revision).
- **Partly fundraising.** WAR still correlates with fundraising (section 1). The gain survives the ensemble, but part
  of what WAR carries overlaps Model B.
- **Not separated from district effects.** For an incumbent, the candidate's strength and a district's persistent
  downballot lean are the same lag. The small open-seat gain suggests the candidate share dominates.
- **Fixed design.** The lag weights (75/25) and the one-coefficient specification were fixed in advance; no
  alternative was searched.